# multi_15m_strat_033

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (216).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `15m` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, ema, dema, compute_normalized_dema, process_ticker, backtest_trades, plot_with_trade_paths, ema, dema, compute_custom_dema, process_ticker, backtest_trades_with_params |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data

In [ ]:
data["Ticker"].unique()

In [ ]:
import pandas as pd
'''
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2023-01-01'
end_date = '2024-12-31'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]'''

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h', offset='15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data = data[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data


In [ ]:
import pandas as pd
'''
# Assuming your DataFrame is called df
data['Date'] = pd.to_datetime(data['Date'])
data = data[data['Date'].dt.time != pd.to_datetime("09:15:00").time()]
data'''

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Parameters
len_dema = 5
base_len = 5
Lu = 90
Su = 10

# Helper functions
def ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def dema(series, length):
    e1 = ema(series, length)
    e2 = ema(e1, length)
    return 2 * e1 - e2

def compute_normalized_dema(group):
    group = group.copy()
    dema_series = dema(group['close'], len_dema)
    base = dema_series.rolling(base_len).mean()
    sd = dema_series.rolling(base_len).std() * 2
    upper_sd = base + sd
    lower_sd = base - sd
    norm_base = 100 * (dema_series - lower_sd) / (upper_sd - lower_sd)

    group['NormBase'] = norm_base
    group['BASE'] = base
    group['UpperSD'] = upper_sd
    group['LowerSD'] = lower_sd

    group['Long'] = (group['NormBase'] > Lu) & (group['close'] > group['LowerSD'])
    group['Short'] = group['NormBase'] < Su

    group['Signal'] = 0
    group.loc[group['Long'] & ~group['Short'], 'Signal'] = 1
    group.loc[group['Short'], 'Signal'] = 2

    return group

# Apply by group
data = data.groupby('Ticker', group_keys=False).apply(compute_normalized_dema)

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    """Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        # Check for open positions and apply exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            # Compute Take Profit (TP) & Stop Loss (SL) prices
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            # Determine exit conditions
            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                # Apply slippage
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)

                # Calculate PnL including brokerage
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                # Store trade details
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Open new position
        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# Main backtesting function
def backtest_trades(data):
    """Runs backtest for all tickers in the dataset using parallel processing."""

    # Trading parameters
    initial_capital = 100000  # ₹1 lakh per trade
    brokerage = 0.001  # 0.1% brokerage
    slippage = 0.000  # 0.01% slippage
    tp_pct = 0.03  # 1.5% take-profit
    sl_pct = 0.005  # 0.5% stop-loss
    end_time = "15:15"
    stop_trading_time = "14:15"

    # Get unique tickers
    unique_tickers = data['Ticker'].unique()

    # Convert DataFrame to NumPy structured array for performance
    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
            .itertuples(index=False, name=None)),
        dtype=dtype
    )

    # Run backtest in parallel
    results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time) for ticker in tqdm(unique_tickers))

    # Flatten results into DataFrame
    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Example usage
# data = pd.read_csv("your_data.csv")  # Load dataset
data['Date'] = pd.to_datetime(data['Date'])  # Ensure Date column is in datetime format
tradebook = backtest_trades(data)
tradebook.to_csv("/content/drive/MyDrive/tradebook.csv", index=False)
tradebook

In [ ]:
data

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'  # or 'notebook', or 'browser'
''''
def plot_with_trade_paths(df, tradebook, ticker_name):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    ticker_df = df[df['Ticker'] == ticker_name]

    trades = tradebook[tradebook['Ticker'] == ticker_name].copy()
    trades['Entry Time'] = pd.to_datetime(trades['Entry Time'])
    trades['Exit Time'] = pd.to_datetime(trades['Exit Time'])

    fig = go.Figure()

    # Plot price and indicators
    fig.add_trace(go.Scatter(x=ticker_df['Date'], y=ticker_df['close'], mode='lines', name='Close Price', line=dict(color='black')))
    fig.add_trace(go.Scatter(x=ticker_df['Date'], y=ticker_df['BASE'], mode='lines', name='BASE', line=dict(color='blue', dash='dash')))
    fig.add_trace(go.Scatter(x=ticker_df['Date'], y=ticker_df['UpperSD'], mode='lines', name='Upper SD', line=dict(color='green', dash='dot')))
    fig.add_trace(go.Scatter(x=ticker_df['Date'], y=ticker_df['LowerSD'], mode='lines', name='Lower SD', line=dict(color='red', dash='dot')))
    fig.add_trace(go.Scatter(x=ticker_df['Date'], y=ticker_df['NormBase'], mode='lines', name='NormBase', line=dict(color='purple')))

    # Plot entries and exits with connecting lines
    for _, row in trades.iterrows():
        entry_color = 'green' if row['Direction'] == 'long' else 'red'
        exit_color = 'black'
        line_color = 'green' if row['PnL'] > 0 else 'red'

        # Entry marker
        fig.add_trace(go.Scatter(
            x=[row['Entry Time']],
            y=[row['Entry Price']],
            mode='markers',
            marker=dict(symbol='triangle-up' if row['Direction'] == 'long' else 'triangle-down', size=10, color=entry_color),
            name=f"{row['Direction'].capitalize()} Entry",
            showlegend=False
        ))

        # Exit marker
        fig.add_trace(go.Scatter(
            x=[row['Exit Time']],
            y=[row['Exit Price']],
            mode='markers',
            marker=dict(symbol='x', size=10, color=exit_color),
            name=f"{row['Direction'].capitalize()} Exit",
            showlegend=False
        ))

        # Connecting line
        fig.add_trace(go.Scatter(
            x=[row['Entry Time'], row['Exit Time']],
            y=[row['Entry Price'], row['Exit Price']],
            mode='lines',
            line=dict(color=line_color, width=2),
            name='PnL Path',
            showlegend=False
        ))

    fig.update_layout(
        title=f"{ticker_name} | Price, Indicators & Trades",
        xaxis_title='Date',
        yaxis_title='Price / Indicator',
        height=800,
        hovermode='x unified'
    )

    fig.show()
plot_with_trade_paths(data, tradebook, ticker_name='ADANIENT')'''


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from itertools import product
'''
# -------------------------------------------------------
# Parameter Grid for Hyperparameter Tuning
# -------------------------------------------------------
param_grid = {
    'len_dema': [5],
    'base_len': [5, 10],
    'Lu': [70, 80, 90, 100],
    'Su': [5, 10],
    'tp_pct': [0.03, 0.05],
    'sl_pct': [0.005, 0.02]
}

param_combinations = list(product(
    param_grid['len_dema'],
    param_grid['base_len'],
    param_grid['Lu'],
    param_grid['Su'],
    param_grid['tp_pct'],
    param_grid['sl_pct']
))

# -------------------------------------------------------
# Indicator Calculations
# -------------------------------------------------------
def ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def dema(series, length):
    e1 = ema(series, length)
    e2 = ema(e1, length)
    return 2 * e1 - e2

# -------------------------------------------------------
# Strategy Signal Generator
# -------------------------------------------------------
def compute_custom_dema(group, len_dema, base_len, Lu, Su):
    group = group.copy()
    dema_series = dema(group['close'], len_dema)
    base = dema_series.rolling(base_len).mean()
    sd = dema_series.rolling(base_len).std() * 2
    upper_sd = base + sd
    lower_sd = base - sd
    norm_base = 100 * (dema_series - lower_sd) / (upper_sd - lower_sd)

    group['NormBase'] = norm_base
    group['Long'] = (norm_base > Lu) & (group['close'] > lower_sd)
    group['Short'] = norm_base < Su
    group['Signal'] = 0
    group.loc[group['Long'] & ~group['Short'], 'Signal'] = 1
    group.loc[group['Short'], 'Signal'] = 2

    return group

# -------------------------------------------------------
# Trade Execution and Backtesting Logic
# -------------------------------------------------------
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        # Exit condition
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                trades.append([
                    ticker, position["entry_time"], timestamp, entry_price, exit_price,
                    pnl, exit_reason, cumulative_pnl, direction
                ])
                del open_positions[ticker]
                continue

        # Entry condition
        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {
                "entry_time": timestamp, "entry_price": entry_price,
                "direction": direction, "qty": qty
            }

    return trades

# -------------------------------------------------------
# Backtest for Given Parameters
# -------------------------------------------------------
def backtest_trades_with_params(data, tp_pct, sl_pct):
    initial_capital = 100000
    brokerage = 0.001
    slippage = 0.0001
    end_time = "15:15"
    stop_trading_time = "14:15"

    unique_tickers = data['Ticker'].unique()

    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
             .itertuples(index=False, name=None)),
        dtype=dtype
    )

    results = Parallel(n_jobs=-1)(
        delayed(process_ticker)(
            ticker, data_np[data_np['Ticker'] == ticker],
            initial_capital, brokerage, slippage,
            tp_pct, sl_pct, end_time, stop_trading_time
        ) for ticker in tqdm(unique_tickers)
    )

    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

# -------------------------------------------------------
# Drawdown Calculation
# -------------------------------------------------------
def calculate_drawdown(cumulative_pnl):
    peak = cumulative_pnl.expanding(min_periods=1).max()
    drawdown = peak - cumulative_pnl
    return drawdown.max()

# -------------------------------------------------------
# Single Parameter Combination Execution
# -------------------------------------------------------
def single_run(len_dema, base_len, Lu, Su, tp_pct, sl_pct):
    temp_data = data.copy()
    temp_data = temp_data.groupby('Ticker', group_keys=False).apply(
        compute_custom_dema, len_dema=len_dema, base_len=base_len, Lu=Lu, Su=Su
    )

    tradebook = backtest_trades_with_params(temp_data, tp_pct, sl_pct)

    if tradebook.empty:
        return {
            "len_dema": len_dema, "base_len": base_len,
            "Lu": Lu, "Su": Su, "tp_pct": tp_pct, "sl_pct": sl_pct,
            "TotalPnL": 0, "MaxDrawdown": 0, "PnL/DD": 0
        }

    total_pnl = tradebook['PnL'].sum()
    max_dd = calculate_drawdown(tradebook['Cumulative PnL'])
    pnl_dd_ratio = total_pnl / max_dd if max_dd != 0 else 0

    print(f"Params: len_dema={len_dema}, base_len={base_len}, Lu={Lu}, Su={Su}, tp={tp_pct}, sl={sl_pct} | "
          f"PnL: {total_pnl:.2f}, Drawdown: {max_dd:.2f}, Ratio: {pnl_dd_ratio:.2f}")

    return {
        "len_dema": len_dema, "base_len": base_len,
        "Lu": Lu, "Su": Su, "tp_pct": tp_pct, "sl_pct": sl_pct,
        "TotalPnL": total_pnl, "MaxDrawdown": max_dd, "PnL/DD": pnl_dd_ratio
    }

# -------------------------------------------------------
# Full Grid Search Execution
# -------------------------------------------------------
results = [single_run(ld, bl, lu, su, tp, sl)
           for ld, bl, lu, su, tp, sl in tqdm(param_combinations)]

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("PnL/DD", ascending=False)
results_df.to_csv("grid_search_results.csv", index=False)

print("Top 5 Results:")
print(results_df.head())
'''